In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r"C:\Customer_Revenue_Intelligence\data\processed\online_retail_clean.csv",
                 parse_dates=['InvoiceDate'])

rfm = pd.read_csv(r"C:\Customer_Revenue_Intelligence\data\processed\rfm_segments.csv")

print(f"Clean dataset loaded: {len(df):,} rows")
print(f"RFM dataset loaded: {len(rfm):,} customers")
print(f"Scipy version: {stats.__version__}")


Clean dataset loaded: 779,407 rows
RFM dataset loaded: 5,878 customers


AttributeError: module 'scipy.stats' has no attribute '__version__'

In [2]:
import scipy
print(f"Clean dataset loaded: {len(df):,} rows")
print(f"RFM dataset loaded: {len(rfm):,} customers")
print(f"Scipy version: {scipy.__version__}")

Clean dataset loaded: 779,407 rows
RFM dataset loaded: 5,878 customers
Scipy version: 1.13.1


In [3]:
champion_revenue = rfm[rfm['Segment'] == 'Champion']['Monetary']
non_champion_revenue = rfm[rfm['Segment'] != 'Champion']['Monetary']

t_stat, p_value = stats.ttest_ind(champion_revenue, non_champion_revenue)

print("=== T-TEST: CHAMPION vs NON-CHAMPION REVENUE ===")
print(f"Champion customers: {len(champion_revenue):,}")
print(f"Non-Champion customers: {len(non_champion_revenue):,}")
print(f"Champion mean revenue: £{champion_revenue.mean():,.2f}")
print(f"Non-Champion mean revenue: £{non_champion_revenue.mean():,.2f}")
print(f"Difference: £{champion_revenue.mean() - non_champion_revenue.mean():,.2f}")
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.10f}")
print(f"\nInterpretation:")
if p_value < 0.05:
    print(f"  p-value ({p_value:.6f}) < 0.05 — STATISTICALLY SIGNIFICANT")
    print(f"  The difference in revenue between Champions and non-Champions")
    print(f"  is NOT due to random chance. It is a real, meaningful difference.")
else:
    print(f"  p-value ({p_value:.6f}) >= 0.05 — NOT statistically significant")
    print(f"  Cannot conclude the difference is meaningful.")

=== T-TEST: CHAMPION vs NON-CHAMPION REVENUE ===
Champion customers: 1,297
Non-Champion customers: 4,581
Champion mean revenue: £9,143.86
Non-Champion mean revenue: £1,203.93
Difference: £7,939.93

T-statistic: 17.9522
P-value: 0.0000000000

Interpretation:
  p-value (0.000000) < 0.05 — STATISTICALLY SIGNIFICANT
  The difference in revenue between Champions and non-Champions
  is NOT due to random chance. It is a real, meaningful difference.


In [4]:
uk_aov = df[df['Country'] == 'United Kingdom'].groupby('Invoice')['Revenue'].sum()
non_uk_aov = df[df['Country'] != 'United Kingdom'].groupby('Invoice')['Revenue'].sum()

t_stat2, p_value2 = stats.ttest_ind(uk_aov, non_uk_aov)

print("=== T-TEST: UK vs NON-UK ORDER VALUE ===")
print(f"UK orders: {len(uk_aov):,}")
print(f"Non-UK orders: {len(non_uk_aov):,}")
print(f"UK mean order value: £{uk_aov.mean():,.2f}")
print(f"Non-UK mean order value: £{non_uk_aov.mean():,.2f}")
print(f"Difference: £{non_uk_aov.mean() - uk_aov.mean():,.2f}")
print(f"\nT-statistic: {t_stat2:.4f}")
print(f"P-value: {p_value2:.10f}")
print(f"\nInterpretation:")
if p_value2 < 0.05:
    print(f"  p-value ({p_value2:.6f}) < 0.05 — STATISTICALLY SIGNIFICANT")
    print(f"  Non-UK customers place statistically larger orders than UK customers.")
    print(f"  This is a real difference, not random variation.")
else:
    print(f"  p-value ({p_value2:.6f}) >= 0.05 — NOT statistically significant")
    print(f"  Cannot conclude the difference is meaningful.")

=== T-TEST: UK vs NON-UK ORDER VALUE ===
UK orders: 33,541
Non-UK orders: 3,428
UK mean order value: £429.00
Non-UK mean order value: £870.94
Difference: £441.93

T-statistic: -18.2063
P-value: 0.0000000000

Interpretation:
  p-value (0.000000) < 0.05 — STATISTICALLY SIGNIFICANT
  Non-UK customers place statistically larger orders than UK customers.
  This is a real difference, not random variation.


In [5]:
observed_counts = rfm['Segment'].value_counts().values
expected_counts = np.full(len(observed_counts), len(rfm) / len(observed_counts))

chi2_stat, p_value3 = stats.chisquare(observed_counts, expected_counts)

print("=== CHI-SQUARE TEST: RFM SEGMENT DISTRIBUTION ===")
print(f"\nObserved vs Expected segment counts:")
seg_counts = rfm['Segment'].value_counts().reset_index()
seg_counts.columns = ['Segment', 'Observed']
seg_counts['Expected'] = round(len(rfm) / len(seg_counts), 1)
seg_counts['Difference'] = seg_counts['Observed'] - seg_counts['Expected']
print(seg_counts.to_string())
print(f"\nTotal customers: {len(rfm):,}")
print(f"Number of segments: {len(seg_counts)}")
print(f"Expected per segment if equal: {len(rfm)/len(seg_counts):.1f}")
print(f"\nChi-square statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value3:.10f}")
print(f"\nInterpretation:")
if p_value3 < 0.05:
    print(f"  p-value ({p_value3:.6f}) < 0.05 — STATISTICALLY SIGNIFICANT")
    print(f"  Customer distribution across segments is NOT random.")
    print(f"  Customers genuinely cluster into distinct behavioural groups.")
else:
    print(f"  p-value ({p_value3:.6f}) >= 0.05 — NOT statistically significant")
    print(f"  Cannot confirm segments are meaningfully distinct.")

=== CHI-SQUARE TEST: RFM SEGMENT DISTRIBUTION ===

Observed vs Expected segment counts:
              Segment  Observed  Expected  Difference
0            Champion      1297     587.8       709.2
1      About to Sleep       880     587.8       292.2
2                Lost       776     587.8       188.2
3      Loyal Customer       603     587.8        15.2
4  Potential Loyalist       535     587.8       -52.8
5         Hibernating       532     587.8       -55.8
6        New Customer       443     587.8      -144.8
7     Needs Attention       393     587.8      -194.8
8             At Risk       223     587.8      -364.8
9           Promising       196     587.8      -391.8

Total customers: 5,878
Number of segments: 10
Expected per segment if equal: 587.8

Chi-square statistic: 1659.4039
P-value: 0.0000000000

Interpretation:
  p-value (0.000000) < 0.05 — STATISTICALLY SIGNIFICANT
  Customer distribution across segments is NOT random.
  Customers genuinely cluster into distinct behavio

In [6]:
df['Month'] = df['InvoiceDate'].dt.month
df['Is_Q4'] = df['Month'].isin([10, 11, 12])

q4_orders = df[df['Is_Q4']].groupby('Invoice')['Revenue'].sum()
non_q4_orders = df[~df['Is_Q4']].groupby('Invoice')['Revenue'].sum()

t_stat4, p_value4 = stats.ttest_ind(q4_orders, non_q4_orders)

print("=== T-TEST: Q4 vs NON-Q4 ORDER VALUE ===")
print(f"Q4 orders: {len(q4_orders):,}")
print(f"Non-Q4 orders: {len(non_q4_orders):,}")
print(f"Q4 mean order value: £{q4_orders.mean():,.2f}")
print(f"Non-Q4 mean order value: £{non_q4_orders.mean():,.2f}")
print(f"Difference: £{q4_orders.mean() - non_q4_orders.mean():,.2f}")
print(f"\nQ4 total revenue: £{df[df['Is_Q4']]['Revenue'].sum():,.2f}")
print(f"Non-Q4 total revenue: £{df[~df['Is_Q4']]['Revenue'].sum():,.2f}")
print(f"Q4 revenue as % of total: {round(df[df['Is_Q4']]['Revenue'].sum()/df['Revenue'].sum()*100,2)}%")
print(f"\nT-statistic: {t_stat4:.4f}")
print(f"P-value: {p_value4:.10f}")
print(f"\nInterpretation:")
if p_value4 < 0.05:
    print(f"  p-value ({p_value4:.6f}) < 0.05 — STATISTICALLY SIGNIFICANT")
    print(f"  Q4 revenue spike is NOT seasonal noise — it is a real pattern.")
else:
    print(f"  p-value ({p_value4:.6f}) >= 0.05 — NOT statistically significant")
    print(f"  Q4 revenue difference cannot be confirmed as meaningful.")

=== T-TEST: Q4 vs NON-Q4 ORDER VALUE ===
Q4 orders: 12,996
Non-Q4 orders: 23,973
Q4 mean order value: £474.19
Non-Q4 mean order value: £467.70
Difference: £6.49

Q4 total revenue: £6,162,555.27
Non-Q4 total revenue: £11,212,248.98
Q4 revenue as % of total: 35.47%

T-statistic: 0.4378
P-value: 0.6614989988

Interpretation:
  p-value (0.661499) >= 0.05 — NOT statistically significant
  Q4 revenue difference cannot be confirmed as meaningful.


In [7]:
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['Month'] = df['InvoiceDate'].dt.month
df['Is_Q4'] = df['Month'].isin([10, 11, 12])

monthly_revenue = df.groupby(['YearMonth', 'Is_Q4'])['Revenue'].sum().reset_index()

q4_monthly = monthly_revenue[monthly_revenue['Is_Q4'] == True]['Revenue']
non_q4_monthly = monthly_revenue[monthly_revenue['Is_Q4'] == False]['Revenue']

t_stat5, p_value5 = stats.ttest_ind(q4_monthly, non_q4_monthly)

print("=== T-TEST: Q4 vs NON-Q4 MONTHLY REVENUE VOLUME ===")
print(f"Q4 months in dataset: {len(q4_monthly)}")
print(f"Non-Q4 months in dataset: {len(non_q4_monthly)}")
print(f"Q4 average monthly revenue: £{q4_monthly.mean():,.2f}")
print(f"Non-Q4 average monthly revenue: £{non_q4_monthly.mean():,.2f}")
print(f"Difference: £{q4_monthly.mean() - non_q4_monthly.mean():,.2f}")
print(f"Q4 premium: {round((q4_monthly.mean()/non_q4_monthly.mean()-1)*100,2)}% higher than non-Q4")
print(f"\nT-statistic: {t_stat5:.4f}")
print(f"P-value: {p_value5:.6f}")
print(f"\nInterpretation:")
if p_value5 < 0.05:
    print(f"  p-value ({p_value5:.6f}) < 0.05 — STATISTICALLY SIGNIFICANT")
    print(f"  Q4 monthly revenue is significantly higher than non-Q4.")
    print(f"  The seasonal spike is real and statistically proven.")
else:
    print(f"  p-value ({p_value5:.6f}) >= 0.05 — NOT statistically significant")
    print(f"  With only 6 Q4 months vs 19 non-Q4 months, sample too small to confirm.")
    print(f"  The visual pattern exists but cannot be statistically proven with this dataset size.")

=== T-TEST: Q4 vs NON-Q4 MONTHLY REVENUE VOLUME ===
Q4 months in dataset: 7
Non-Q4 months in dataset: 18
Q4 average monthly revenue: £880,365.04
Non-Q4 average monthly revenue: £622,902.72
Difference: £257,462.32
Q4 premium: 41.33% higher than non-Q4

T-statistic: 3.2814
P-value: 0.003274

Interpretation:
  p-value (0.003274) < 0.05 — STATISTICALLY SIGNIFICANT
  Q4 monthly revenue is significantly higher than non-Q4.
  The seasonal spike is real and statistically proven.


In [8]:
print("=" * 65)
print("STATISTICAL VALIDATION SUMMARY")
print("=" * 65)
print(f"\n{'Test':<45} {'P-Value':<12} {'Result'}")
print("-" * 65)
print(f"{'T-Test: Champion vs Non-Champion Revenue':<45} {'~0.000':<12} {'SIGNIFICANT ✓'}")
print(f"{'T-Test: UK vs Non-UK Order Value':<45} {'~0.000':<12} {'SIGNIFICANT ✓'}")
print(f"{'Chi-Square: RFM Segment Distribution':<45} {'~0.000':<12} {'SIGNIFICANT ✓'}")
print(f"{'T-Test: Q4 vs Non-Q4 Order Size (AOV)':<45} {'0.6615':<12} {'NOT significant ✗'}")
print(f"{'T-Test: Q4 vs Non-Q4 Monthly Revenue':<45} {'0.0033':<12} {'SIGNIFICANT ✓'}")
print("-" * 65)
print(f"\nKEY FINDINGS:")
print(f"1. Champions earn £7,939 more than non-Champions — statistically proven")
print(f"2. Non-UK customers place £441 larger orders — statistically proven")
print(f"3. RFM segments reflect real behavioural clusters — statistically proven")
print(f"4. Q4 spike is volume-driven, not AOV-driven — statistically proven")
print(f"5. Q4 generates 41.33% more monthly revenue than non-Q4 — proven")
print(f"\nHONEST NOTE:")
print(f"Test 4 (AOV) returned p=0.66 — NOT significant.")
print(f"We report this honestly. Q4 buyers don't spend more per order.")
print(f"The seasonal effect is purely transactional volume, not basket size.")
print("=" * 65)

STATISTICAL VALIDATION SUMMARY

Test                                          P-Value      Result
-----------------------------------------------------------------
T-Test: Champion vs Non-Champion Revenue      ~0.000       SIGNIFICANT ✓
T-Test: UK vs Non-UK Order Value              ~0.000       SIGNIFICANT ✓
Chi-Square: RFM Segment Distribution          ~0.000       SIGNIFICANT ✓
T-Test: Q4 vs Non-Q4 Order Size (AOV)         0.6615       NOT significant ✗
T-Test: Q4 vs Non-Q4 Monthly Revenue          0.0033       SIGNIFICANT ✓
-----------------------------------------------------------------

KEY FINDINGS:
1. Champions earn £7,939 more than non-Champions — statistically proven
2. Non-UK customers place £441 larger orders — statistically proven
3. RFM segments reflect real behavioural clusters — statistically proven
4. Q4 spike is volume-driven, not AOV-driven — statistically proven
5. Q4 generates 41.33% more monthly revenue than non-Q4 — proven

HONEST NOTE:
Test 4 (AOV) returned p=0.